# Preparación de entrenamiento · raw → refined → experimentos

Esta corrida prepara **v3.50**, usando el índice limpio del 50% y las ocho revisiones humanas
aplicadas en `data_lookup`. YOLO recibe solo candidatos; las detecciones compatibles se reutilizan.
Las versiones v2 y sus experimentos permanecen intactos. Las salidas anteriores, si las hay,
son evidencia histórica; solo una nueva ejecución documenta v3.50.

Primero ejecuta en **CPU hasta «Entrada compacta»**. Después conecta una **GPU**, repite
instalación/verificación/configuración y continúa con YOLO. Solo una sesión escribe los respaldos.
La GPU carga las candidatas compactadas, sin restaurar todo el historial raw.


## Fase 1. Preparación y preprocesamiento de datos

Instala `pettrace-ml[training,notebooks]==0.2.16` directamente desde PyPI. No hay perfiles de
requirements ni archivos que subir manualmente; el wheel y cada preparación registran su procedencia.
Usa un kernel nuevo.
No actualices paquetes durante una preparación activa.

Fuentes: [Dogs of the World v1](https://www.kaggle.com/datasets/lextoumbourou/dogs-world)
y [LCW v1](https://www.kaggle.com/datasets/dseidli/lcwlabeled-cats-in-the-wild).
El porcentaje se calcula sobre las fotos originales antes de filtrar, con semilla 42 y muestras anidadas.
La selección, mínimo de cuatro fotos distintas, deduplicación y cuarentena son decisiones de esta investigación.
Las etiquetas de origen no equivalen a identidades verificadas. E1/E2 permanecen congelados.


In [ ]:
import sys

if sys.version_info[:2] != (3, 13) or sys.platform != "linux":
    raise RuntimeError("This E3 Colab profile requires Linux and Python 3.13")
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks]==0.2.16"


In [1]:
import os
import torch
import pettrace
from pettrace.tracking import source_identity
from pettrace.data.workflow import TrainingPreparation

if pettrace.__version__ != "0.2.16":
    raise RuntimeError("Install pettrace-ml[training,notebooks]==0.2.16 before continuing")
code_identity = source_identity()
if code_identity["build"].get("version") != "0.2.16":
    raise RuntimeError("Use the published pettrace-ml==0.2.16 wheel")
print("Package:", pettrace.__version__, "· commit:", code_identity["git_sha"])
print("Source SHA256:", code_identity["source_sha256"])
print("Working tree changes in build:", code_identity["git_dirty"])

print("GPU disponible:", torch.cuda.is_available(), "· staging no necesita GPU")


Package: 0.2.14 · commit: 4745adf3ac55e6b1cffff03c08a894b153c1a8a8
Source SHA256: a15ddf99952a0d76f6e6b3cd1a8fb2dccafe3c10d09478be86e3bc69ad5b0b26
Working tree changes in build: False
GPU disponible: True · staging no necesita GPU


### Recuperar la base refinada

Las versiones existentes son inmutables: se cargan y verifican; no se regeneran con otras reglas.
Si una versión falta, se reutilizan las cachés de adquisición y YOLO de preparaciones anteriores.
Los experimentos solo necesitan la raíz `prepared`, que contiene imágenes relativas, manifiesto,
splits, auditoría y hashes. No necesitan las fuentes ni los ZIP.

El índice corresponde al 50%: 43.091 candidatas antes de reservar val/test y aplicar calidad.
Las 44 etiquetas revisadas se agrupan en ocho; seis grupos superan el mínimo antes de YOLO.


In [2]:
import json
from pathlib import Path
from time import perf_counter
from IPython.display import Image, Markdown, display

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
SOURCE_SAMPLE_PCT = 50  # 20, 50 o 100 de las fotos originales, antes de filtrar.
CANDIDATE_INDEX = Path("/content/drive/MyDrive/PetTrace/data-lookup/9340a0dedd9077304df35ae68839df0a5c670708fc1dae76f7d0214f7b4bab65")
if not (CANDIDATE_INDEX / "lookup.json").is_file():
    raise FileNotFoundError("Falta el índice revisado de data_lookup en Drive")
preparation = TrainingPreparation(
    drive_root="/content/drive/MyDrive/PetTrace", work_dir="/content",
    source_percent=SOURCE_SAMPLE_PCT, workers=4, screening_provider="cuda",
    candidate_index=CANDIDATE_INDEX, version="v3.50")
prepared = preparation.restore()
print(f"Refined {preparation.version}: " + ("verificada; limpieza omitida." if prepared else "pendiente; requiere entrada compacta."))


Mounted at /content/drive
Refined v3.50: pendiente; requiere entrada compacta.


### Entrada compacta · ejecutar en CPU una vez

Reúne las candidatas y detecciones compatibles en Drive. Si faltan fotos locales, la primera
compactación puede recuperar bloques históricos grandes. Al terminar, cambia a GPU; conserva
`work_dir="/content"` y el mismo índice. Las salidas antiguas reflejan el flujo anterior.


In [ ]:
if prepared is None and not torch.cuda.is_available():
    staged = preparation.stage(detector_dir=os.environ.get("E3_DETECTOR_DIR"))
    print("Entrada respaldada. Conecta la GPU, repite setup y continúa con YOLO.")
elif prepared is None:
    print("GPU: se exigirá la entrada compacta antes de restaurar fotos.")


### Aplicar YOLO a los candidatos del índice

La lógica vive en `pettrace.data.workflow.TrainingPreparation`: restaurar, compactar, preparar y publicar son métodos
reutilizables fuera del notebook. Esta ejecución usa solo la entrada compacta; no adquiere ZIPs.
Las detecciones CPU compatibles se conservan; las nuevas CUDA tienen caché separada.

Se usa el detector congelado de E2 ([YOLO](https://github.com/ultralytics/ultralytics)), con el mismo
preprocesamiento, NMS y umbrales. [ONNX Runtime CUDA](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html)
y el [plugin oficial CUDA 12 v0.3.0](https://github.com/microsoft/onnxruntime/releases/tag/plugin-ep-cuda/v0.3.0)
fijado por SHA256 permiten usar GPU conservando el runtime de la caché CPU.

Antes del lote se comprueba la asignación de convoluciones a CUDA y se comparan hasta 16 gatos y 16 perros,
seleccionados por SHA256 y fuera de las identidades públicas reservadas. Se exige igual número, especie y orden
de detecciones; tolerancias propias: puntuación ≤ 1e-4, coordenadas ≤ 0,1 píxeles.
La muestra debe coincidir y resultar más rápida en GPU; de lo contrario se detiene y guarda el diagnóstico.
Esto **no prueba igualdad en todas las fotos**. `gpu-validation.json` conserva la prueba y sus tiempos.

El índice se verifica contra el inventario antes de YOLO. Las detecciones se reutilizan por hash
de imagen, modelo, configuración y entorno; cambiar la etiqueta por una fusión no cambia esa clave.

La limpieza acepta exactamente un animal de la especie esperada; no certifica identidades.
Los splits públicos **val/test v1** se conservan. Los respaldos periódicos verifican bloques en Drive;
una desconexión brusca conserva solo los respaldos confirmados.

Un índice SHA256 en memoria comparte verificaciones entre etapas. Cada archivo se vuelve a leer si cambia
su tamaño, mtime, ctime o identidad en disco. El índice se descarta al finalizar; otra sesión verifica
de nuevo. La referencia v1 completa se carga localmente sin consultar HF cuando coincide con el commit
y fingerprint fijados. Los bloques transferidos desde Drive conservan su verificación completa.


La ampliación conserva las exclusiones manuales, cuarentena de etiquetas contradictorias,
duplicados exactos/cercanos y mínimo de fotos por identidad. Puede cambiar el conjunto retenido
al descubrir conflictos nuevos; las muestras **raw** sí son anidadas.


In [3]:
started = perf_counter()
prepared = preparation.prepare(detector_dir=os.environ.get("E3_DETECTOR_DIR"))
print("Dataset local:", preparation.version_path)
print("Fingerprint:", prepared.fingerprint)
print(f"Preparación: {(perf_counter() - started) / 60:.1f} min")


Drive: bloque restaurado (44125 archivos).
Drive: bloque restaurado (2773 archivos).
Drive: bloque restaurado (2836 archivos).
Drive: bloque restaurado (2729 archivos).
Drive: bloque restaurado (8371 archivos).
Drive: bloque restaurado (9377 archivos).
Drive: bloque restaurado (9087 archivos).
Drive: bloque restaurado (8154 archivos).
Drive: bloque restaurado (390 archivos).
Drive: 87842 archivos verificados; respaldo listo.
Entrada compacta: 43091 candidatas · 4.07 GiB
Drive: 42893 archivos verificados; respaldo listo.
Índice previo: 347296 fotos → 42893 candidatas a YOLO
YOLO: 42893 fotos · caché CPU 17527 · caché CUDA 25366 · pendientes 0
CPU/GPU: 32 fotos coinciden · aceleración medida 5.71×
Drive: respaldo confirmado (1 archivos, 01793725617249654729-60dee83b8470494cbde9486533136a9e.tar).
YOLO: 1000/42893 revisadas · inferencias nuevas GPU 0
YOLO: 2000/42893 revisadas · inferencias nuevas GPU 0
YOLO: 3000/42893 revisadas · inferencias nuevas GPU 0
YOLO: 4000/42893 revisadas · infe

## Fase 2. Representación y entrenamiento

Aquí no se entrena ningún modelo. La salida es una base refinada: manifiesto, identidades,
splits, imágenes y procedencia. YOLO filtra calidad y número de animales; no reconoce individuos.
El entrenamiento del MLP corresponde al notebook E3 que cargue esta nueva versión.

## Fase 3. Evaluación y análisis

### Auditoría antes y después

El reporte conserva cada exclusión, cajas y puntuaciones YOLO, hashes, procedencia y configuración.
El índice previo no garantiza el mínimo final de fotos: se vuelve a comprobar tras filtrar.
`identity-review.json` conserva las ocho confirmaciones y los alias; las demás etiquetas de origen
siguen sin validación humana. El benchmark val/test v1 se conserva sin cambios.


In [4]:
audit = json.loads((prepared.root / "audit.json").read_text())
screening = json.loads((prepared.root / "animal-screening.json").read_text())
summary = prepared.summary
print("Fotos candidatas:", summary["candidate_photos"], "→ retenidas:", summary["retained_photos"])
print("Identidades candidatas:", summary["candidate_identities"], "→ retenidas:", summary["retained_identities"])
print("YOLO:", json.dumps(screening["counts"], indent=2))
print("Motivos de exclusión:", json.dumps(summary["exclusion_reasons"], indent=2))
print("Duplicados eliminados:", len(audit["removed_duplicates"]))
print("Identidades en cuarentena:", len(audit["conflicting_identities_quarantined"]))
lines = ["| Especie / split | Identidades | Fotos |", "|---|---:|---:|"]
for split, count in summary["counts"].items():
    lines.append(f"| {split} | {count['identities']} | {count['photos']} |")
display(Markdown("\n".join(lines)))
print(summary["limitation"])

print("Procesamiento por fuente:", json.dumps(summary.get("processing", {}), indent=2))


Fotos candidatas: 347296 → retenidas: 27518
Identidades candidatas: 196874 → retenidas: 6484
YOLO: {
  "multiple_animals": 4474,
  "no_animal_detected": 1015,
  "single_expected_animal": 36547,
  "unexpected_species": 857
}
Motivos de exclusión: {
  "decode_error": 2,
  "exact_duplicate": 1377,
  "identity_review_pending": 36048,
  "insufficient_distinct_photos": 273221,
  "known_ambiguous_identity": 3,
  "multiple_animals": 4474,
  "near_copy_review_pending": 2833,
  "no_animal_detected": 1015,
  "reserved_public_identity_or_image": 498,
  "unexpected_species": 857
}
Duplicados eliminados: 0
Identidades en cuarentena: 19379


| Especie / split | Identidades | Fotos |
|---|---:|---:|
| cat.test | 34 | 162 |
| cat.train | 4974 | 21057 |
| cat.val | 34 | 163 |
| dog.test | 26 | 115 |
| dog.train | 1390 | 5911 |
| dog.val | 26 | 110 |

Exploratory pilot: pending decisions use publisher identity labels without individual visual approval. Technical filtering and a visual sample do not verify all identities or sessions. A single detection does not verify identity. Occlusion, missed animals and false detections remain possible; these thresholds have not been calibrated on this pilot. Sample agreement is not proof of identical outputs on every image.
Procesamiento por fuente: {
  "dogs-world": {
    "excluded_photos": 150924,
    "exclusion_reasons": {
      "exact_duplicate": 407,
      "identity_review_pending": 26428,
      "insufficient_distinct_photos": 120800,
      "known_ambiguous_identity": 3,
      "missing_or_ambiguous_identity": 0,
      "multiple_animals": 881,
      "near_copy_review_pending": 1456,
      "no_animal_detected": 518,
      "reserved_public_identity_or_image": 267,
      "unexpected_species": 164
    },
    "note": "Processing includes metadata/held-out rejection before image filtering; v1 val/

### Muestra visual reproducible

Se muestran cuatro identidades por especie después del filtrado. La selección queda guardada en `visual-sample.json`; permite inspeccionar problemas visibles sin aprobar automáticamente el resto del dataset.


In [ ]:
sample = json.loads((prepared.root / "visual-sample.json").read_text())
for item in sample["sample"]:
    print(item["animal"])
    display(Image(filename=str(prepared.root / item["sheet"])))


### Consumir desde E3

En una sesión nueva, E3 restaura solo `prepared` de los respaldos de Drive y carga el recibo
`v3.50.json` de esta preparación. Verifica hashes antes de entrenar; no ejecuta esta preparación.
Si se usa HF, se fijan commit y fingerprint de la publicación. PetTrace v4 se evalúa por separado.

La optimización conserva la metodología, pero registra una implementación distinta en la procedencia.
Este cuaderno no prueba una aceleración determinada en Colab: depende del disco, Drive y hardware asignado.


In [ ]:
print("E3_PREPARED_DATASET =", preparation.version_path)
if release:
    print(json.dumps({
        "HF_TRAINING_DATASET_REPO": release["repo"],
        "HF_TRAINING_DATASET_VERSION": release["version"],
        "HF_TRAINING_DATASET_REVISION": release["commit"],
        "HF_TRAINING_DATASET_FINGERPRINT": release["fingerprint"],
    }, indent=2))

print("Versión refinada lista:", preparation.version, "· fingerprint:", prepared.fingerprint)
